# 课程 5 · 文本进入模型：token 账与 embedding

先观察一个真实例子，再提出解释，最后用代码或估算验证。结果和预测不一样时，就从那个差异继续查。

**这一课替谁做什么。** 设想你在帮一家公司评估“把合同交给大模型问答”的方案。对方一定会问：一份合同算多少 token、大概花多少钱、会不会超出模型一次能读的长度；还会问方案里常说的 “embedding、向量库” 到底是什么。

**我们手里有什么。** Qwen 公开的真实分词器文件（只有切分规则，没有模型参数，本地运行不花钱）；一张随机生成的 8 行小表，用来看清查表这一步；从 Qwen 模型卡和配置文件核对过的真实参数数字。

**还缺什么。** 本课不运行完整模型，也不调用付费 API。所以这里数出的 token 不是任何厂商的账单，用到的价格都是标明的教学假设。

**想得到什么。** 能自己测出一类材料的 token 密度，写出带假设和单位的估算；能用两三句话讲清模型内部的 embedding 表，并把它和 RAG 用的文本向量区分开。

三站路线：**A 分词与 token 账 → B 编号进入模型后先查表 → C 同叫 embedding 的两样东西**。A 约 40 分钟（前半你已做完），B、C 合计约 30 分钟，可以分两次学。

<details><summary>本课任务与积分规则（展开查看）</summary>

必做：任务 1 分段/整串计数对照（你已完成，原样保留）；任务 2 token 密度函数；两道估算；结尾一次面试解释。

原来的 one-hot 等价证明、改表行的代码、显式 one-hot 内存账和补齐平均，移到 [optional-deep-dive.ipynb](optional-deep-dive.ipynb)。它们偏向模型实现，解决方案面试一般不考，主课和作业都不依赖。想看底层时再打开。

- 本课有一个小游戏「猜 token 数」，在第 1 节后面。每轮 5 题，从 10 段方案里常见的材料中随机抽，再来一轮会换题；最好成绩记在本地。
- 积分不是掌握程度：操作最多 25 分（分词观察 5、猜数打完一轮 10、查表预测 10），两项代码各 20 分，必做共 65；之前玩旧版两关得到的分数照常保留。选做 Notebook 另有 80。重复运行不重复加分，答错不扣分。
- 代码检查会换输入再测；提示使用情况由你自报。估算和解释需要讨论，程序不自动判定。
- 动画没有声音，会遵守系统的“减少动态效果”设置。本地进度保存在本课 `.learning-progress.json`，不上传。

</details>

## 准备实验台

沿用 **llm-from-zero** 内核。下面两格负责加载和显示，已经写好；若之前运行过旧版，请先重启内核，再从这里运行。后面的代码任务和估算留给你完成。**按顺序运行，不要点“全部运行”**，因为 TODO 仍故意未填写。

<details><summary>环境与显示检查：组件不出现时再展开</summary>

第一格打印 Python 路径并读取本地分词器，完成后只显示一行加载结果；第二格准备小游戏和本地记录，只打印一行确认。

`ipywidgets` 已安装到项目环境。若提示缺依赖，在项目根目录运行 `uv sync`，重启内核后再运行准备格。分词器约 11.4 MB，无模型权重，不需要云 GPU。

若只见 `VBox(...)`、空白或加载组件失败，确认 VS Code 的 Jupyter 扩展可用、内核选对并重新运行；也可继续下方普通代码实验。动画被编辑器过滤时不影响 Python 检查和本地记录。显示故障不等于你的答案错误。

</details>


In [13]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

candidates = [Path.cwd() / "05-token-embedding", Path.cwd()]
COURSE_DIR = next((p for p in candidates if (p / "tokenizer_tools.py").exists()), None)
if COURSE_DIR is None:
    raise RuntimeError("请从项目根目录或 05-token-embedding 目录运行。")
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))
from IPython.display import display
from lab_visuals import TokenizerLoadPanel

# 显示一行加载结果；五步加载证据可以展开查看。
loader = TokenizerLoadPanel()
display(loader.widget)
tokenizer = loader.load()
torch.set_printoptions(precision=3, sci_mode=False)
print("Python:", sys.executable)
print("已登记的 token 数（含新增项）:", tokenizer.get_vocab_size(with_added_tokens=True))
print("只加载本地分词器；没有加载语言模型，也没有 API 请求。")


Python: /Users/yorkchen/文稿 - 陈尧的MacBook Pro/VibeCoding/llm-from-zero/.venv/bin/python
已登记的 token 数（含新增项）: 151669
只加载本地分词器；没有加载语言模型，也没有 API 请求。


In [14]:
from learning_lab import LessonLab
from token_games import TokenGames

# 只负责小游戏、代码检查与本地记录，不提供下面任务的答案。
lab = LessonLab(tokenizer, COURSE_DIR / ".learning-progress.json")
games = TokenGames(tokenizer, lab.progress, motion=lambda: lab.motion.value)
print("关卡和记录已准备好。")

关卡和记录已准备好。


### 1．文字、token 和 token ID 是三样东西

我们看到的是文字，分词器切出的是 token，词表给每个 token 一个编号。下面同时看这三样，别急着把编号当成含义。

先预测下面三句话的 token 数是否相同，再运行。`len(text)` 数 Python 字符串中的 Unicode 码点，并不是所有情况下肉眼看到的“字数”。

本课用到的接口：
- `tokenizer.encode(text, add_special_tokens=False)`：编码普通文本，这里不让后处理额外添加标记。
- 结果的 `.ids`：token 编号；`.tokens`：词表里的内部显示形式。
- `tokenizer.decode(ids, skip_special_tokens=False)`：把整串编号还原成文字。

**token ID 是词表编号。** 编号相邻不能推出含义相近，编号大也不能推出重要程度高。后面还需要查 embedding 表。


In [15]:
samples = ["Hello, world!", "你好，世界！", "AI助手帮我查天气。"]
for text in samples:
    encoded = tokenizer.encode(text, add_special_tokens=False)
    restored = tokenizer.decode(encoded.ids, skip_special_tokens=False)
    print("\n原文:", repr(text))
    print("Python字符数 / token数:", len(text), len(encoded.ids))
    print("token ID:", encoded.ids)
    print("词表内部显示:", encoded.tokens)
    print("整串还原:", restored)
    assert restored == text



原文: 'Hello, world!'
Python字符数 / token数: 13 4
token ID: [9707, 11, 1879, 0]
词表内部显示: ['Hello', ',', 'Ġworld', '!']
整串还原: Hello, world!

原文: '你好，世界！'
Python字符数 / token数: 6 4
token ID: [108386, 3837, 99489, 6313]
词表内部显示: ['ä½łå¥½', 'ï¼Į', 'ä¸ĸçķĮ', 'ï¼ģ']
整串还原: 你好，世界！

原文: 'AI助手帮我查天气。'
Python字符数 / token数: 10 6
token ID: [15469, 110498, 108965, 32876, 104307, 1773]
词表内部显示: ['AI', 'åĬ©æīĭ', 'å¸®æĪĳ', 'æŁ¥', 'å¤©æ°Ķ', 'ãĢĤ']
整串还原: AI助手帮我查天气。


<details><summary>为什么卡片上的字符看起来奇怪？</summary>

你可能看到词表内部显示不像汉字。这份分词器采用字节级 BPE：用字节表示文本，再依据已有合并规则形成 token，并为它们编号。一个 token 可能包含整个词、词的一部分，或一个字符的一部分字节。它不是按空格简单切词。

词表内部的字符表示不直接等于最终展示文字。单独解码某一个 token 可能出现替代字符 `�`；完整 UTF-8 字节要组合起来才能还原。所以这里验证**整串 decode**，不把逐个 decode 的结果拼起来当作原文。

本课只处理普通文本，不实现 BPE 的训练算法。

</details>


#### 小游戏 · 猜 token 数

给客户估算费用，第一步就是回答“这些材料大概多少 token”。先练手感：每轮 5 题，都是方案里常见的材料，比如合同、客服 FAQ、英文手册、工具返回的 JSON、代码，每轮随机抽。

每题先凭感觉报一个 token 数，回车揭晓：句子会碎成 token 块，尺子标出你猜的和真实的差距，再按这一段的比例换算成“1 万个字符约多少 token”。误差在 1 个或 10% 以内算命中，3 分；30% 以内算接近，1 分；连续命中有连击。

打完一轮会画出每类材料的真实比例和你猜的比例，看看自己在哪类材料上偏得最多。再来一轮会换题。

揭晓时顺便留意 token 块的边界：空格跟着前一个词还是后一个词？数字、英文单词、汉字各自怎么切？第 3 节会用到。

想在浏览器里看，而不是在格子里看：在项目根目录运行 `uv run python 05-token-embedding/web/server.py`，打开 http://127.0.0.1:8765 。同一份本地分词器，可以贴任何一段材料；5 题游戏也在那一页，最好成绩写进同一个进度文件。函数和估算还是在下面的格子里做。

In [ ]:
display(games.guess())

### 2．你刚问过的换行，会进入模型输入吗？

下面三段字符串分别包含：一个真正换行、反斜杠加字母 n、两个真正换行。先预测它们是否有相同的编码。

`repr(text)` 是方便检查的显示方式：它把真正换行显示成转义写法。**显示方式与字符串实际内容需要区分。** 运行输出后，结合长度与编码作判断。


In [16]:
text_variants = ["第一段\n第二段", r"第一段\n第二段", "第一段\n\n第二段"]
for text in text_variants:
    encoded = tokenizer.encode(text, add_special_tokens=False)
    print("repr:", repr(text))
    print("字符数 / token数 / IDs:", len(text), len(encoded.ids), encoded.ids)


repr: '第一段\n第二段'
字符数 / token数 / IDs: 7 5 [99363, 37474, 198, 99802, 37474]
repr: '第一段\\n第二段'
字符数 / token数 / IDs: 8 5 [99363, 37474, 1699, 99802, 37474]
repr: '第一段\n\n第二段'
字符数 / token数 / IDs: 8 5 [99363, 37474, 271, 99802, 37474]


### 3．真正要你判断的地方：分段数能直接相加吗？

程序把提示词拆成了“规则”“材料”“问题”几段。有人打算分别数 token，再把数量加起来，认为一定等于最终请求的文本数量。

RAG 把文档切块后分别计数，也会碰到同一个问题。回想游戏里的 token 块：空格常跟着后面的词，几个字母可能合成一个 token。如果切点正好落在一个 token 中间，会怎样？一次相同能证明总是相同吗？一次不同又能说明什么？

**任务 1：实现一个完整比较函数。** 输入是若干文本片段；返回：
- 拼接后的原文 `text`；
- 整串编码 `joined_ids`；
- 各段分别编码再拼接的 `piece_ids`；
- 两种 token 数 `joined_count`、`piece_count`。

要用的写法：`"".join(["ab", "cd"])` 得到 `"abcd"`；`items.extend([2, 3])` 把两个元素追加进列表；返回字典用 `return {"名称": value}`。不需要考虑并发、网络或特殊聊天标记。


<details><summary>线索 1：先别看数字，看看操作顺序</summary>

我们把两条路径各说一遍：整串路径先拼文字再 encode；分段路径先分别 encode，再连接 ID。保留两组 ID，你会比只看数量得到更多证据。

</details>

<details><summary>线索 2：先组织好记录</summary>

先得到拼接文本和整串 ID。再建一个空列表，遍历每段，把该段编码后的 ID 用 extend 加进去。最后把文本、两组 ID 和各自长度放进字典。函数仍由你完整写出来。

</details>

<details><summary>局部例子：append 和 extend 的差别</summary>

`items = [1]` 后执行 `items.extend([2, 3])` 得到 `[1, 2, 3]`。这里想连接的是编号序列，别把每段编号变成嵌套列表。这个例子解决的是列表操作，编码与比较仍要由你完成。

</details>


In [17]:
def compare_encodings(parts, tokenizer):
    # TODO 1：完整实现两条路径并返回约定字段。
    text = "".join(parts)
    piece_ids = []
    joined_ids =  tokenizer.encode(text, add_special_tokens=False).ids
    for part in parts:
        piece_ids.extend(tokenizer.encode(part, add_special_tokens=False).ids)
    joined_count = len(joined_ids)
    piece_count = len(piece_ids)
    return {"joined_ids":joined_ids, "piece_ids":piece_ids,"text":text,"joined_count":joined_count,"piece_count":piece_count}

comparison = compare_encodings(["nor", "mal"], tokenizer)
print(comparison)
for key, value in comparison.items():
    print(key, ":", value)
assert comparison["text"] == "normal"
assert tokenizer.decode(comparison["joined_ids"], skip_special_tokens=False) == "normal"
assert tokenizer.decode(comparison["piece_ids"], skip_special_tokens=False) == "normal"


{'joined_ids': [8252], 'piece_ids': [44707, 14649], 'text': 'normal', 'joined_count': 1, 'piece_count': 2}
joined_ids : [8252]
piece_ids : [44707, 14649]
text : normal
joined_count : 1
piece_count : 2


In [18]:
# 完成上方函数并运行后，再检查。改过函数后可以直接再次点击按钮。
display(lab.checkpoint("compare", lambda: compare_encodings))


**逐项记录：**

1. 两种 token 数分别是：_1个和两个___。ID 列表是否相同：_不相同___。
2. 两种 ID 都能还原同一段文字，为什么仍可能不同？__我觉得可能是因为在英文里面hel和lo可能就有单独含义，因此被单独分词了。_。
3. 再选一组片段验证，写出片段和结果：_换成其他的也会___。
4. 如果要检查最终输入是否超过一个明确的 token 预算，应在处理流程的哪一步计数？_应该在call模型之前___。

当前实验只测这份 Qwen 分词器的**普通文本**。它没有自动执行聊天模板，没有自动加入角色、工具定义等完整请求内容。`add_special_tokens=False` 也不代表把原文里已经存在的特殊标记强制当普通字符处理。

因此这些计数不能直接替代 DeepSeek 或 TokenHub 的账单；服务端返回的 usage 和对应服务的计费规则才是核账依据。这里学的是计数机制，不是用别家的分词器“校准”你的账号。


### 4．同样的意思，中文和英文谁更费 token？

方案里估算成本时，常用一个比例：每个字大约几个 token。这个比例从哪里来、能不能套到所有材料上？游戏里你已经看到不同材料差得很多；这里用同一条合同条款的中英文，再加一段代码和一段 JSON，把测量写成能反复用的函数。

先写下两个预测：**按每个字符算**，中文和英文谁的 token 比例高？**按同一条意思算**，谁的 token 总数多？

**任务 2：写 `token_density(tokenizer, samples)`。** 后面的估算要用实测比例，而不是凭印象；把测量写成函数，客户换一批材料时只改输入就能重测。

- 输入 `samples` 是字典：名字 → 文本。
- 返回列表，顺序与输入一致；每项是字典，含 `name`、`chars`（`len(text)`）、`tokens`（用 `add_special_tokens=False` 编码后的 ID 个数）、`tokens_per_char`（`tokens / chars`）。
- 某段文本为空时 `raise ValueError(...)`：没有字符，比例没有意义，不能悄悄返回 0。

要用到的写法：

```python
for name, text in {"a": "猫", "b": "狗"}.items():  # 同时取到键和值；字典保持插入顺序
    print(name, text)
rows = []
rows.append({"name": "a", "chars": 1})            # 往列表里加一个字典
raise ValueError("说明原因")                        # 明确停止非法输入
```

<details><summary>线索 1：比例的分子和分母</summary>

分子是这段文本编码后的 token 个数，分母是 Python 字符数。空文本要在做除法之前检查。

</details>

<details><summary>线索 2：循环里做三件事</summary>

每一段：先检查是否为空，再编码计数，最后把四个字段放进一个字典加入结果列表。循环结束后返回列表。

</details>

In [21]:
def token_density(tokenizer, samples):
    # TODO 2：逐段测量字符数、token 数和每字符 token 数；空文本要报错。
    rows = []
    for name, text in samples.items():
        if not text:
            raise ValueError("文本为空")
        else:
            tokens = tokenizer.encode(text, add_special_tokens=False).ids
            chars = len(text)
            tokens_per_char = len(tokens) / chars
            rows.append({"name": name, "chars": chars, "tokens": len(tokens), "tokens_per_char": tokens_per_char})
    return rows
    
samples = {
    "中文合同条款": "本合同自双方签字盖章之日起生效，有效期为三年。乙方应在每季度结束后十个工作日内提交服务报告；每逾期一日，按合同总额的千分之一支付违约金。",
    "同一条款的英文": "This agreement takes effect on the date it is signed and sealed by both parties and remains valid for three years. The vendor shall submit a service report within ten business days after the end of each quarter; for each day of delay, the vendor shall pay liquidated damages equal to 0.1% of the total contract amount.",
    "Python 代码": "def late_fee(total, days):\n    return total * 0.001 * days\n",
    "JSON 记录": '{"contract_id": "HT-2026-0158", "term_years": 3, "report_deadline_days": 10}',
}

density = token_density(tokenizer, samples)
for row in density:
    print(f"{row['name']:<10} 字符 {row['chars']:>4}   token {row['tokens']:>4}   每字符 {row['tokens_per_char']:.3f}")

中文合同条款     字符   68   token   41   每字符 0.603
同一条款的英文    字符  318   token   66   每字符 0.208
Python 代码  字符   59   token   20   每字符 0.339
JSON 记录    字符   76   token   34   每字符 0.447


In [22]:
display(lab.checkpoint("density", lambda: token_density))

**逐项记录：**

1. 每字符比例：中文条款 __0.603__，英文条款 _0.208___。同一条款的 token 总数：中文 __41__，英文 _66___。
2. “中文比英文更费 token”这句话，按字符比和按同样的意思比，结论分别是什么？__按照字符比例来看，中文确实更费token，但是按照意思来看的话，要表达同样的意思，中文不一定比英文贵__。
3. 代码和 JSON 的比例落在哪里？如果客户材料里大量是表格或 JSON，用中文条款的比例去估，会偏高还是偏低？__会偏高__。
4. 这个比例是用 Qwen 分词器、几十个字测出来的。拿它去估别家服务的费用之前，还缺哪两项验证？__1、大规模的预测2、其他的分词器__。

<details><summary>做完再看：这个比例在方案里怎么用</summary>

比例只适合做第一轮粗估：确定数量级、判断会不会超出窗口、比较几种方案。样本要从客户的真实材料里抽，不同类型的材料分开测。正式报价时，用目标服务返回的 usage，或该服务提供的计数工具，对一批样本实测后再定。

</details>

### 5．真正发出去的请求，比正文多了什么？

调用对话 API 时，你发的是一组消息（系统提示、用户问题、历史回答……）。服务端会把它们拼成一段带角色标记的文本，再分词交给模型。Qwen 用 `<|im_start|>`、`<|im_end|>` 这类标记包住每条消息，这些标记在词表里各是一个 token。

下面这几行负责按这个格式把两条消息拼起来，我已经写好。它是**按公开格式手写的近似**：Qwen 官方模板还可能按思考模式等加入其他内容；别家服务的格式不同，而且在服务端完成，你看不到。先预测：拼好后比只数正文多几个 token？

In [23]:
messages = [
    {"role": "system", "content": "你是合同审查助手，只根据材料回答。"},
    {"role": "user", "content": "乙方逾期提交报告要承担什么责任？"},
]
content_tokens = sum(len(tokenizer.encode(m["content"], add_special_tokens=False).ids) for m in messages)

# 每条消息包上角色标记，最后留出助手开始回答的位置。
wrapped = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages) + "<|im_start|>assistant\n"
wrapped_encoding = tokenizer.encode(wrapped, add_special_tokens=False)

print("只数正文:", content_tokens)
print("按消息格式拼好后:", len(wrapped_encoding.ids))
print("前 12 个 token:", wrapped_encoding.tokens[:12])
assert tokenizer.decode(wrapped_encoding.ids, skip_special_tokens=False) == wrapped

只数正文: 19
按消息格式拼好后: 32
前 12 个 token: ['<|im_start|>', 'system', 'Ċ', 'ä½łæĺ¯', 'åĲĪåĲĮ', 'å®¡æŁ¥', 'åĬ©æīĭ', 'ï¼Į', 'åıª', 'æł¹æį®', 'æĿĲæĸĻ', 'åĽŀçŃĶ']


**记录：**

- 多出的 token 有几个，来自哪里？每多一条消息，大约多几个？__13个，来自于<|im_start|>等__。
- 第 4 课里“附上历史再追问”为什么更贵？用这里的结果补一句：_相当于system的信息更多了___。
- 工具定义（告诉模型有哪些函数可调用）通常也放进同一段文本。Agent 方案里工具很多时，会影响什么？_输入的token就更多了___。

还有一件事要记住：**模型一次能处理的长度（上下文窗口）要同时装下输入和输出。** Qwen3-0.6B 模型卡写明原生窗口为 32,768 tokens；它的长文本说明也是按“输入加输出的总长度”来讲的。所以估算时，输入之外还要给回答预留空间。

### 6．估算：一份合同放进模型，要多少 token、放不放得下、多少钱？

这一题是本课的必做估算。数字不难，难在把假设、单位和没算进去的东西都写清楚。客户追问“这个数怎么来的”，你要能一项一项指出来。

**场景与假设**（全部是本题设定，不是实测，也不是任何厂商报价）：

| 项目 | 假设值 | 来历 |
|---|---|---|
| 合同长度 | 30 页，每页约 800 个汉字 | 题目假设；实际要从客户材料里抽样统计 |
| 每字符 token 数 | 用你在任务 2 测出的中文条款比例 | 本课实测，Qwen 分词器 |
| 系统提示、问题和消息标记 | 共约 300 tokens | 题目假设 |
| 给回答预留 | 2,000 tokens | 题目假设 |
| 上下文窗口 | 32,768 tokens，输入加输出共用 | Qwen3-0.6B 模型卡 |
| 价格 | 输入 2 元/百万 tokens，输出 8 元/百万 tokens | **教学假设** |
| 用量 | 每天 200 次问答，每次都带上整份合同 | 题目假设 |

下面的格子已经放好变量。请自己写出计算：合同 token 数、单次输入 token 数、是否放得下、单次费用（输出按预留的 2,000 全部用满，作为保守估计）、每天费用。

In [ ]:
zh_ratio = 0.603            # ← 从 density 结果里取中文合同条款的 tokens_per_char
pages, chars_per_page = 30, 800
overhead_tokens = 300
reserve_output = 2_000
window = 32_768
price_in = 2 / 1_000_000   # 教学假设：元 / 输入 token
price_out = 8 / 1_000_000  # 教学假设：元 / 输出 token
daily_requests = 200

# TODO 估算：算出下面这些量并打印，单位写在变量名或打印文字里。
# contract_tokens, input_tokens, fits, cost_per_request, cost_per_day
contract_tokens = pages * chars_per_page * zh_ratio


**我的估算：**

- 公式（用文字或符号写清楚每一步）：____。
- 结果：合同约 ____ tokens；单次输入约 ____ tokens；加上预留输出后 ____（能 / 不能）放进 32,768；单次约 ____ 元；每天约 ____ 元。
- 未计入的项目（至少写三项）：____。

**换个条件：** 合同变成 80 页，其他不变。还放得下吗？如果放不下，你能想到哪几种处理办法，各有什么代价？____。

<details><summary>提示 1：先列公式的骨架</summary>

合同 token = 页数 × 每页字数 × 每字符 token 数。单次输入 = 合同 token + 其他开销。是否放得下要比较“输入 + 预留输出”和窗口。费用 = 输入 token × 输入单价 + 输出 token × 输出单价，每天再乘以次数。

</details>

<details><summary>提示 2：数量级自查</summary>

30 页合同的输入应在一万多 token；单次费用在几分钱，每天在十元以内的量级。如果你的结果差了一千倍，先检查单价有没有换成“元 / token”。80 页的情况要和窗口比一比。

</details>

<details><summary>提示 3：80 页放不下时的思路方向</summary>

可以想三个方向：换窗口更大的模型；把合同切成几段分别问，再汇总；只把和问题相关的几段找出来再放进请求。最后这种就是后面 RAG 课程要做的事。比较时想一想费用、等待时间，以及会不会漏掉信息。

</details>

## B · 编号进入模型后，第一步做什么？

![词表矩阵与查表结果矩阵的行列含义](assets/lesson5-axes.svg)

这一段只求讲清机制：不写矩阵代码，只看一张小表怎样被使用，再用真实模型的配置算一笔账。完整推导留在选做 Notebook。

模型拿到 token ID 后，第一步是**查表**：有一张表 `E`，词表里每个 token 对应一行，每行 D 个数。输入里有几个 token，就按 ID 取出几行，排成 `[T, D]` 的矩阵，交给后续网络。

- V 是表的行数，由词表决定；T 是这次输入的 token 个数；D 是每行有几个数，由模型设计决定。**输入变长，只有 T 变，表不变。**
- 你在 bigram 课里写过 `one-hot @ W`。one-hot 的一行只有一个 1，乘上表以后正好留下那一行，所以“乘 one-hot”和“直接按编号取行”结果相同。实际实现都直接取行，不去造一个几乎全是 0 的大矩阵。

下面用 8 个人为定义的 token 和一张随机 8×4 的表来看。**这张表没有训练，数值没有含义**，只用来看清查表这个动作。

In [23]:
toy_vocab = ["猫", "追", "狗", "，", "看", "星", "空", "。"]
V, D = len(toy_vocab), 4
generator = torch.Generator(device="cpu").manual_seed(7)
E = torch.randn(V, D, generator=generator, dtype=torch.float32, device="cpu")
ids = torch.tensor([0, 1, 2, 3, 0, 4, 5, 6, 7], dtype=torch.long, device="cpu")

print("ID顺序:", ids.tolist())
print("对应文字:", "".join(toy_vocab[i] for i in ids.tolist()))
print("E形状:", tuple(E.shape))
print(E)


ID顺序: [0, 1, 2, 3, 0, 4, 5, 6, 7]
对应文字: 猫追狗，猫看星空。
E形状: (8, 4)
tensor([[-0.820,  0.396,  0.899, -1.388],
        [-0.167,  0.285, -0.641, -0.894],
        [ 0.927, -0.536, -1.160, -0.460],
        [ 0.709,  1.013,  0.230,  1.090],
        [-1.583, -0.325,  1.926, -0.330],
        [ 0.198,  0.782,  1.039, -0.725],
        [-0.209, -0.215, -1.816, -0.345],
        [-2.061,  0.674, -1.323, -1.360]])


### 7．看一张表怎样被反复使用

拖动“输入位置”，看它选中表里的哪一行。然后选一行、设一个增量，**先在“预测位置”里写下哪些输入位置会受影响，再点揭晓**。实验台只改它自己的副本，不会动上面的 `E`。

In [18]:
display(lab.embedding(E, ids, toy_vocab))


**记录：**

- 改 ID 0 那一行时，哪些位置变了？为什么改一行会影响多个位置？____。
- 同一个 ID 在不同位置取到的是同一行。这能否说明模型最终对“买了个苹果手机”和“吃了个苹果”里的“苹果”理解一样？____。

<details><summary>提示：查表之后还有什么</summary>

查表只是第一步，它不看上下文。后续网络会让每个位置参考其他位置的信息，再更新自己的表示，下一课的 attention 就做这件事。所以“初始向量相同”和“最终表示相同”是两回事。

</details>

### 8．估算：embedding 表在一个真实模型里占多大？

模型最后还要给词表里每个候选 token 打一个分，才能选出下一个 token（你在 bigram 课里见过“给每个候选打分”）。这一步也需要一张 `V × D` 的矩阵。有的模型让它和输入表**共用同一张表**，只存一份；有的分开存两份。配置文件里的 `tie_word_embeddings` 就记录这一点。

下面的数字都已核对（2026-10-31）：0.6B 来自本课分词器同一固定版本的 `config.json` 与模型卡，8B 来自其 `main` 分支的 `config.json` 与模型卡。模型卡上的参数量四舍五入到 0.01B。

| 模型 | 词表 V | 维度 D | 输入表与输出层共用 | 模型卡：总参数 | 模型卡：不含 embedding |
|---|---|---|---|---|---|
| Qwen3-0.6B | 151,936 | 1,024 | 是 | 0.6B | 0.44B |
| Qwen3-8B | 151,936 | 4,096 | 否 | 8.2B | 6.95B |

请在下面的格子里算：每个模型 embedding 相关的参数量（共用算一张，不共用算两张）、它和模型卡“总参数 − 不含 embedding”的差值是否吻合、占总参数的比例。

In [ ]:
configs = {
    "Qwen3-0.6B": {"V": 151_936, "D": 1024, "tied": True, "card_total": 0.6e9, "card_non_embedding": 0.44e9},
    "Qwen3-8B": {"V": 151_936, "D": 4096, "tied": False, "card_total": 8.2e9, "card_non_embedding": 6.95e9},
}

# TODO 估算：对每个模型打印 embedding 参数量、模型卡差值、占总参数的比例。

**我的结论：**

- 0.6B：embedding 参数约 ____，模型卡差值约 ____，占比约 ____。8B：____、____、____。
- 为什么小模型里 embedding 占比更高？____。
- 按 bfloat16（每个参数 2 字节），0.6B 的这张表占多少 GB（10⁹ 字节）？____。
- 如果为了支持更多语言或行业词把词表扩大一倍，参数量和同一段中文的 token 数，可能分别怎么变？____。

<details><summary>提示：差值为什么能对上</summary>

模型卡的“总参数 − 不含 embedding”，就是 embedding 相关的那部分。共用时只有一张 `V × D`；不共用时输入表和输出层各一张。比较时注意模型卡只保留到 0.01B。

</details>

## C · 同叫 embedding 的两样东西

方案讨论里常听到：“先把文档做 embedding，存进向量库，用户提问时按相似度找回相关段落。”这里的 embedding 和 B 部分那张表是一回事吗？

先做一个实验：用 B 部分的表，把一句话里每个 token 的向量取平均，得到一个“句子向量”。“猫追狗”和“狗追猫”的意思正好相反，先预测这两个平均向量的关系，再运行。

In [ ]:
cat_chases_dog = torch.tensor([0, 1, 2])
dog_chases_cat = torch.tensor([2, 1, 0])
mean_a = E[cat_chases_dog].mean(dim=0)
mean_b = E[dog_chases_cat].mean(dim=0)
torch.testing.assert_close(mean_a, mean_b)
print("两个平均向量:", mean_a, mean_b)


![同叫 embedding 的两样东西](assets/lesson5-two-embeddings.svg)

两个平均向量完全相同。原因有两层：这张表是随机的，没有学过任何意思；平均只看“有哪些 token”，把顺序丢掉了。

RAG 用的是另一种东西：**文本向量模型**（embedding model）。它内部同样先查表，然后经过多层网络，让每个位置结合上下文计算，最后把整段文字汇总成**一个长度固定的向量**。它经过专门训练，目标是让意思相近的文本得到相近的向量。

| | 模型内部的输入表 | 文本向量模型（RAG 用） |
|---|---|---|
| 输入 | 一个 token ID | 一整段文本 |
| 输出 | 每个 token 一行，`[T, D]` | 整段一个向量，长度固定 |
| 看不看上下文 | 不看，同一个 ID 永远是同一行 | 读完整段才输出 |
| 方案里怎么接触 | 一般接触不到，在服务端模型内部 | 单独的接口或本地模型，结果存进向量库 |

本课只亲手看左边这一列。右边的真实调用、文档切块和检索效果评测，放在第 12–15 课。

**记录：**

- 平均之后两句话分不开，丢掉了什么信息？____。
- 客户问：“你们说的 embedding，是不是就是大模型里那张表？”用两三句话回答：____。
- 如果把向量库里用的文本向量模型换成另一家的，库里已经存好的向量还能直接和新问题的向量比较吗？为什么？____。

<details><summary>提示 1：两个向量能比较的前提</summary>

两个向量能比“近不近”，前提是它们由同一个模型、按同一套规则算出来，每一维的含义才对得上。

</details>

<details><summary>提示 2：换模型的实际代价</summary>

换了模型，新问题的向量和旧文档的向量就不在同一套坐标里了，维度也可能不同。想一想：文档有几十万段时，这一步在方案里意味着什么工作量和费用。

</details>

## 用一分钟解释

面试官问：“客户想把几百份中文合同接入大模型问答。你怎么估算 token 和费用？方案里提到的 embedding 又是什么？”

请按四项回答，每项一两句：

- 结论：____。
- 机制：分词、消息格式和窗口怎样决定 token 数；embedding 表和文本向量模型各做什么：____。
- 证据：引用本课的实际结果，比如一次分段计数反例、中英文密度、合同估算的数字：____。
- 局限：本课的数字哪些不能直接当报价，还需要什么验证：____。

In [ ]:
display(lab.reflection_panel())


到这里本课的必做部分就结束了。

- 想看底层实现：[optional-deep-dive.ipynb](optional-deep-dive.ipynb)，包括 shape 推导、one-hot 与查表等价的代码证明、改表行定位、显式 one-hot 的内存账和补齐平均。选做，不影响作业。
- [课后练习](homework.ipynb)：上下文预算函数、新材料的密度与费用估算、embedding 占比迁移、面试追问。
- 下一课：attention，看每个位置怎样根据上下文更新自己的表示。

官方来源：[Qwen3-0.6B 模型卡](https://huggingface.co/Qwen/Qwen3-0.6B)、[固定版本 config.json](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/config.json)、[Qwen3-8B 模型卡](https://huggingface.co/Qwen/Qwen3-8B)、[Qwen3-8B config.json](https://huggingface.co/Qwen/Qwen3-8B/blob/main/config.json)、[Qwen 分词概念](https://qwen.readthedocs.io/en/latest/getting_started/concepts.html#byte-level-byte-pair-encoding)、[Tokenizers 接口](https://huggingface.co/docs/tokenizers/v0.20.3/api/tokenizer)。